# Whisper speech-to-text

English | [简体中文](01_finetune.zh-CN.ipynb)

## Goal

Walk through validated audio, epoch-boundary recovery, held-out evaluation, export/reload and inference using the same implementation as the CLI. The default is an offline CPU smoke run, not evidence of speech quality.

> The smoke backend is a randomly initialized native Whisper with a synthetic tokenizer. Sine waves do not contain their text labels.

## Setup

Use a fresh speech environment: Transformers 4.57.3 cannot be mixed with tracks pinning 5.18.0. The next cell installs dependencies only in Colab. Locally install the matching extra before opening this notebook. Restart a Colab runtime if installing replaces packages already loaded. Keep long experiments on Drive.

In [1]:
import importlib.util
import subprocess
import sys
from pathlib import Path

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    ROOT = Path("/content/advai-notebooks")
    if not ROOT.exists():
        subprocess.run(["git", "clone", "--depth", "1",
                        "https://github.com/advai-x/advai-notebooks.git", str(ROOT)], check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e",
                    f"{ROOT}[stt,notebooks]"], check=True)
else:
    ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents]
                if (path / "pyproject.toml").exists())
print("Repository and speech environment ready.")

Repository and speech environment ready.


### Imports and parameters

The seed, data, budgets and optimizer come from the checked-in YAML. A fresh output directory makes repeated notebook execution safe.

In [2]:
import json
import uuid
import warnings
import torch
from IPython.display import Audio, Markdown, display
from advai_notebooks.stt.config import load_config
from advai_notebooks.stt.data import load_data, prepare_data
from advai_notebooks.stt.model import STTModel
from advai_notebooks.stt.pipeline import run
from advai_notebooks.stt.publish import publish
from advai_notebooks.common.runtime import setup
from transformers.utils import logging

logging.set_verbosity_error()
warnings.filterwarnings("ignore", message="Moving the following attributes.*")
cfg = load_config(ROOT / "configs/stt/smoke.yaml", ROOT)
cfg.output_dir = f"outputs/stt/notebook-{uuid.uuid4().hex[:8]}"
print(json.dumps({"model": cfg.model.id, "seed": cfg.training.seed,
                  "epochs": cfg.training.epochs, "device": cfg.runtime.device}, indent=2))

{
  "model": "tiny",
  "seed": 42,
  "epochs": 2,
  "device": "cpu"
}


## Steps

### 1. Validate and prepare audio

Groups and identical decoded audio cannot cross splits. Clips must be finite mono audio within duration limits. Inspect a bounded manifest preview and listen to the synthetic input.

In [3]:
rows, summary = load_data(cfg)
validator = STTModel(cfg.model, setup(cfg), cfg.root, cfg.training)
prepared = prepare_data(rows, validator, cfg, fit=True)
print(json.dumps({"splits": summary["splits"], "sample_rate": summary["sample_rate"]}, indent=2))
for row in rows[:2]:
    print({key: row[key] for key in ("id", "group_id", "split", "text")})
display(Audio(rows[0]["waveform"], rate=summary["sample_rate"]))
del validator, prepared

{
  "splits": {
    "train": 4,
    "validation": 2,
    "test": 2
  },
  "sample_rate": 16000
}
{'id': 'one', 'group_id': 'recording-1', 'split': 'train', 'text': 'red'}
{'id': 'two', 'group_id': 'recording-2', 'split': 'train', 'text': 'green'}


### 2. Train and deliberately interrupt after epoch one

The pipeline saves a baseline and atomic checkpoints before invoking the callback. This simulates a disconnected runtime at an epoch boundary without losing the optimizer/RNG state.

In [4]:
class TutorialInterruption(Exception):
    pass

def interrupt_after_first_epoch(epoch, state):
    if epoch == 1:
        raise TutorialInterruption("Saved epoch-one checkpoint.")

try:
    run(cfg, callbacks=(interrupt_after_first_epoch,))
except TutorialInterruption as error:
    print(error)
latest = cfg.path(cfg.output_dir) / "latest"
assert (latest / "state.json").is_file()

Saved epoch-one checkpoint.


### 3. Resume and inspect held-out metrics

Resume requires unchanged config, data, source and environment. Validation chooses the lowest metric, including the baseline; test is report-only.

In [5]:
report = run(cfg, resume=latest)
assert report["training"]["epoch"] == cfg.training.epochs
metrics = {"baseline_test": report["baseline_test"]["metrics"],
           "selected_test": report["finetuned_test"]["metrics"],
           "selected_epoch": report["training"]["best_epoch"]}
print(json.dumps(metrics, indent=2))

{
  "baseline_test": {
    "nll": 3.531214952468872,
    "wer": 4.0,
    "cer": 5.625
  },
  "selected_test": {
    "nll": 3.4273252487182617,
    "wer": 1.0,
    "cer": 1.0
  },
  "selected_epoch": 1
}


The values above are a smoke diagnostic, not a benchmark. STT WER/CER can exceed 1 because insertions count; unsegmented Chinese is better assessed with CER. TTS loss is teacher forcing, not a naturalness score. A baseline can remain the selected model.

### 4. Reload and infer

Load the selected export without the original training dataset. No weights or audio are uploaded.

In [6]:
selected = STTModel.load(report['export'], 'cpu')

In [7]:
transcription = selected.transcribe(rows[-1]["audio"])
print({"reference": rows[-1]["text"], "transcription": transcription})

{'reference': 'pink', 'transcription': ''}


## Checks

Confirm metadata/native assets, reload parity and an offline whitelist publication preview. Only publish a voice embedding with explicit speaker permission; it is derived biometric data.

In [8]:
export = Path(report["export"])
assert (export / "advai_metadata.json").is_file()
reloaded = STTModel.load(export, "cpu")
for name, value in selected.model.state_dict().items():
    assert torch.equal(value, reloaded.model.state_dict()[name])
plan = publish(export, "example/stt-smoke", dry_run=True)
names = [item["path"] for item in plan["files"]]
assert not any(name.endswith((".wav", ".jsonl", ".pt")) for name in names)
print({"reload_equal": True, "publish": plan["status"], "files": len(names)})

{'reload_equal': True, 'publish': 'dry_run', 'files': 9}


## Next Steps

See the [full guide](../../docs/stt.md). Copy `configs/stt/hf.yaml` to `configs/local/`, pin the model and use your own consented speech corpus with independent sessions and held-out text. The real preset requires target-hardware validation; do not train a pretrained speech model on these tones. This executed tutorial does not establish GPU fine-tuning, free-Colab fit or real speech quality. Use fixed unseen texts for baseline/selected TTS listening.